# Анализ ошибок гибридного классификатора ЭКГ (`DS2` Hold-out Test)
Визуализация ложноположительных и ложноотрицательных срабатываний модели (`HybridECGClassifier`): форма QRS-комплекса (256 отсчётов, R-пик на отсчёте 96) + табличные признаки ритма (`pre_RR`, `post_RR`, `local_RR`, `ratio_RR`).

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
import yaml

from src.contracts import IDX_TO_LABEL
from src.train import InMemoryECGDataset, load_npz_safely, resolve_dataset_path
from src.evaluate import load_model_from_checkpoint, predict_on_loader

In [ ]:
config_path = ROOT_DIR / 'configs/config.yaml'
with open(config_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f) or {}

data_cfg = cfg.get('data', {})
model_cfg = cfg.get('model', {})
train_cfg = cfg.get('training', cfg.get('train', {}))

ckpt_dir = Path(train_cfg.get('checkpoint_dir', 'checkpoints'))
if not ckpt_dir.is_absolute():
    ckpt_dir = ROOT_DIR / ckpt_dir
ckpt_file = ckpt_dir / 'best_model.pt'

test_npz_path = resolve_dataset_path(data_cfg, 'test_file', 'ds2_test.npz')
if not test_npz_path.is_absolute():
    test_npz_path = ROOT_DIR / test_npz_path

r_peak_idx = int(data_cfg.get('r_peak_index', data_cfg.get('samples_before_r', 96)))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, ckpt = load_model_from_checkpoint(ckpt_file, model_cfg, device)

signals, rr_features, labels, patient_ids = load_npz_safely(test_npz_path)
loader = DataLoader(InMemoryECGDataset(signals, rr_features, labels), batch_size=256, shuffle=False)

y_true, y_pred, y_probs = predict_on_loader(model, loader, device)
print(f'Всего ударов в DS2: {len(y_true):,} | Ошибок: {np.sum(y_true != y_pred):,}')

In [ ]:
def plot_top_confident_errors(true_cls_idx: int, pred_cls_idx: int, top_k: int = 4):
    """Отрисовывает top_k примеров, где модель увереннее всего перепутала true_cls с pred_cls."""
    mask = (y_true == true_cls_idx) & (y_pred == pred_cls_idx)
    error_indices = np.where(mask)[0]
    if len(error_indices) == 0:
        print(f'Ошибок вида {IDX_TO_LABEL[true_cls_idx]} -> {IDX_TO_LABEL[pred_cls_idx]} не найдено.')
        return

    confidences = y_probs[error_indices, pred_cls_idx]
    top_idx = error_indices[np.argsort(-confidences)[:top_k]]

    fig, axes = plt.subplots(1, len(top_idx), figsize=(4.5 * len(top_idx), 3.8), dpi=130)
    if len(top_idx) == 1:
        axes = [axes]

    for ax, idx in zip(axes, top_idx):
        sig = signals[idx].squeeze()
        rr = rr_features[idx]
        conf = y_probs[idx, pred_cls_idx]
        ax.plot(sig, color='#1f77b4', lw=1.4)
        ax.axvline(r_peak_idx, color='red', linestyle='--', alpha=0.5, label=f'R-пик ({r_peak_idx})')
        ax.set_title(
            f'Пациент #{patient_ids[idx]} | P({IDX_TO_LABEL[pred_cls_idx]})={conf:.2f}\n'
            f'pre={rr[0]:.2f}s, post={rr[1]:.2f}s, ratio={rr[3]:.2f}',
            fontsize=9
        )
        ax.legend(loc='upper right', fontsize=8)
        ax.grid(alpha=0.3)

    fig.suptitle(
        f'Ошибки классификации: Истина = {IDX_TO_LABEL[true_cls_idx]} -> Предсказание = {IDX_TO_LABEL[pred_cls_idx]}',
        fontsize=12,
        fontweight='bold'
    )
    plt.tight_layout()
    plt.show()

# Разбор самых частых клинических путаниц:
plot_top_confident_errors(true_cls_idx=1, pred_cls_idx=0, top_k=4)  # S принят за норму N
plot_top_confident_errors(true_cls_idx=2, pred_cls_idx=0, top_k=4)  # V принят за норму N